# Demo: Embedding Client (`shared/embeddings.py`)

**Purpose** — Validation notebook for `shared/embeddings.py`. Demonstrates that all
six models load correctly, produce embeddings of the expected shape and normalization,
and pass three semantic sanity checks before any Lens experiment is run.

**Thesis relevance** — §7.1 (model selection) and §7.2 (analysis plan). The three
checks below are the empirical preconditions of the Lens analyses:

| # | Check | Lens relevance |
|---|-------|----------------|
| 1 | Correct shape and L2 normalization | cosine sim = dot product (all Lenses) |
| 2 | Semantic coherence (EN + ZH) | models encode legal structure, not noise (Lens I–V) |
| 3 | Within-tradition RDM consistency | 3+3 robustness design is meaningful (§7.1 D1) |

**References**
- Reimers & Gurevych (2019). Sentence-BERT. *EMNLP 2019*. arXiv:1908.10084
- Kriegeskorte et al. (2008). Representational similarity analysis. *Front. Systems Neurosci.*, 2, 4.
- Nili et al. (2014). A toolbox for representational similarity analysis. *PLoS Comp. Biol.*, 10(4).

In [8]:
import sys
import time
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr

# ── Path setup ───────────────────────────────────────────────────────────────
# Works whether the notebook is run from experiments/ or experiments/notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent  # → experiments/
sys.path.insert(0, str(ROOT))

from shared.embeddings import EmbeddingClient  # noqa: E402

CONFIG = ROOT / "models" / "config.yaml"
assert CONFIG.exists(), f"Config not found: {CONFIG}"
print(f"Config: {CONFIG}")

Config: /Users/gpuzio/Desktop/CODE/THESIS/experiments/models/config.yaml


## 1. Model inventory

Instantiate the client and print the full model registry. Confirms that `config.yaml`
is parsed correctly and all six models are registered with the expected labels,
language codes, and embedding dimensions.

In [9]:
client = EmbeddingClient(CONFIG, device="cpu")

print(f"{'Label':<22} {'Group':<8} {'Lang':<6} {'Dim':<6}  Model ID")
print("-" * 85)
for spec in client.weird_specs:
    print(f"{spec.label:<22} {'WEIRD':<8} {spec.lang:<6} {spec.dim:<6}  {spec.id}")
for spec in client.sinic_specs:
    print(f"{spec.label:<22} {'Sinic':<8} {spec.lang:<6} {spec.dim:<6}  {spec.id}")

Label                  Group    Lang   Dim     Model ID
-------------------------------------------------------------------------------------
BGE-EN-large           WEIRD    en     1024    BAAI/bge-large-en-v1.5
E5-large               WEIRD    en     1024    intfloat/e5-large-v2
FreeLaw-EN             WEIRD    en     768     freelawproject/modernbert-embed-base_finetune_512
BGE-ZH-large           Sinic    zh     1024    BAAI/bge-large-zh-v1.5
Text2vec-large-ZH      Sinic    zh     1024    GanymedeNil/text2vec-large-chinese
Dmeta-ZH               Sinic    zh     768     DMetaSoul/Dmeta-embedding-zh


## 2. Output shape and L2 normalization

All output arrays must have shape `(N, dim)` and be L2-normalized (unit norm per row).
After L2 normalization, cosine similarity reduces to the dot product:

$$\cos(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v} \quad \text{iff} \quad \|\mathbf{u}\| = \|\mathbf{v}\| = 1$$

This property is exploited by all five Lens analyses, which operate on cosine
distance matrices rather than raw inner products.

Test run on one representative model per tradition.

In [10]:
checks = [
    ("BAAI/bge-large-en-v1.5", ["criminal", "contract", "jurisdiction"], 1024),
    ("BAAI/bge-large-zh-v1.5", ["刑事", "合同", "管轄權"],              1024),
]

for model_id, texts, expected_dim in checks:
    vecs = client.embed(texts, model_id)
    norms = np.linalg.norm(vecs, axis=1)

    assert vecs.shape == (len(texts), expected_dim), (
        f"{model_id}: expected shape ({len(texts)}, {expected_dim}), got {vecs.shape}"
    )
    assert np.allclose(norms, 1.0, atol=1e-5), (
        f"{model_id}: embeddings not unit-norm (norms = {norms})"
    )

    print(f"✓ {model_id}")
    print(f"  shape = {vecs.shape}  |  norms = {norms.round(6)}")

✓ BAAI/bge-large-en-v1.5
  shape = (3, 1024)  |  norms = [1. 1. 1.]
✓ BAAI/bge-large-zh-v1.5
  shape = (3, 1024)  |  norms = [1. 1. 1.]


## 3. Semantic coherence — WEIRD models (English)

A necessary condition for the RDM analyses (Lens I, II) is that each model
encodes domain structure: legal terms must be **more similar to each other**
than to semantically unrelated (Swadesh) terms.

**Metric**: mean pairwise cosine similarity (= dot product after L2 normalization).  
**Probe sets**:
- Legal EN: 10 core terms from `legal_terms.json`
- Non-legal EN: 10 terms from `swadesh_control.json` (basic vocabulary)

**Expected**: `mean(legal ↔ legal) > mean(legal ↔ Swadesh)` for all 3 WEIRD models.

In [11]:
legal_en = [
    "criminal", "contract", "jurisdiction",
    "habeas corpus", "mens rea", "due process",
    "statute", "liability", "equity", "appeal",
]
swadesh_en = [
    "hand", "water", "fire", "sun", "fish",
    "eye", "stone", "blood", "bone", "road",
]

print(f"{'Model':<22}  legal↔legal   legal↔Swadesh    gap")
print("-" * 65)

for spec in client.weird_specs:
    vecs = client.embed(legal_en + swadesh_en, spec.id)
    lv = vecs[:len(legal_en)]
    sv = vecs[len(legal_en):]

    # Off-diagonal upper triangle for legal↔legal
    idx = np.triu_indices(len(legal_en), k=1)
    mean_ll = (lv @ lv.T)[idx].mean()
    mean_ls = (lv @ sv.T).mean()
    gap = mean_ll - mean_ls

    status = "✓" if gap > 0 else "✗"
    print(f"{status} {spec.label:<20}  {mean_ll:.4f}          {mean_ls:.4f}          {gap:+.4f}")
    assert gap > 0, f"{spec.label}: legal cohesion check failed"

Model                   legal↔legal   legal↔Swadesh    gap
-----------------------------------------------------------------
✓ BGE-EN-large          0.6284          0.5783          +0.0501
✓ E5-large              0.8055          0.7802          +0.0253
✓ FreeLaw-EN            0.4201          0.4133          +0.0068


## 4. Semantic coherence — Sinic models (Chinese)

Same check applied to the three Sinic models.

**Probe sets**:
- Legal ZH: Traditional Chinese equivalents of the 10 EN legal terms above
- Non-legal ZH: Traditional Chinese Swadesh items

In [12]:
legal_zh = [
    "刑事", "合同", "管轄權",
    "人身保護令", "犯罪意圖", "正當程序",
    "法規", "法律責任", "衡平法", "上訴",
]
swadesh_zh = [
    "手", "水", "火", "太陽", "魚",
    "眼睛", "石頭", "血", "骨頭", "道路",
]

print(f"{'Model':<22}  legal↔legal   legal↔Swadesh    gap")
print("-" * 65)

for spec in client.sinic_specs:
    vecs = client.embed(legal_zh + swadesh_zh, spec.id)
    lv = vecs[:len(legal_zh)]
    sv = vecs[len(legal_zh):]

    idx = np.triu_indices(len(legal_zh), k=1)
    mean_ll = (lv @ lv.T)[idx].mean()
    mean_ls = (lv @ sv.T).mean()
    gap = mean_ll - mean_ls

    status = "✓" if gap > 0 else "✗"
    print(f"{status} {spec.label:<20}  {mean_ll:.4f}          {mean_ls:.4f}          {gap:+.4f}")
    assert gap > 0, f"{spec.label}: legal cohesion check failed"

Model                   legal↔legal   legal↔Swadesh    gap
-----------------------------------------------------------------
✓ BGE-ZH-large          0.4498          0.3333          +0.1165
✓ Text2vec-large-ZH     0.2954          0.2192          +0.0763
✓ Dmeta-ZH              0.5631          0.4483          +0.1148


## 5. Within-tradition RDM consistency (WEIRD)

The 3+3 robustness design (§7.1 D1) requires that the three WEIRD models produce
**structurally similar pairwise distance geometries** for the same set of terms.
If the three models fundamentally disagree on which terms are close and which
are distant, the within-tradition consistency layer of the analysis collapses.

Procedure:
1. Compute the **Representational Dissimilarity Matrix (RDM)** for each WEIRD model
   on a 10-term probe set. RDM entry $(i,j) = 1 - \cos(\mathbf{v}_i, \mathbf{v}_j)$.
2. Extract the upper triangle of each RDM as a flat vector.
3. Compute pairwise **Spearman ρ** between the three upper-triangle vectors.

**Expected**: ρ > 0.5 for all off-diagonal pairs. Values below 0.5 would flag
a within-tradition architecture confound that would need to be reported in §7.2.

**Reference**: Kriegeskorte et al. (2008). RSA connects the branches of systems
neuroscience. *Frontiers in Systems Neuroscience*, 2, 4.

In [13]:
probe_en = [
    "criminal", "contract", "jurisdiction",
    "habeas corpus", "mens rea", "due process",
    "statute", "liability", "equity", "appeal",
]

rdms: dict[str, np.ndarray] = {}
tri_idx = np.triu_indices(len(probe_en), k=1)

for spec in client.weird_specs:
    vecs = client.embed(probe_en, spec.id)
    dist = 1.0 - (vecs @ vecs.T)  # cosine distance (in [0, 2] after norm)
    rdms[spec.label] = dist[tri_idx]

labels = list(rdms)
print("Spearman ρ — pairwise RDM correlation (WEIRD models, 10-term probe set)\n")
header_width = 24
print(f"{'':>{header_width}}", end="")
for l in labels:
    print(f"{l:>{header_width}}", end="")
print()
for l1 in labels:
    print(f"{l1:>{header_width}}", end="")
    for l2 in labels:
        r, _ = spearmanr(rdms[l1], rdms[l2])
        print(f"{r:>{header_width}.4f}", end="")
    print()

print()
all_ok = True
for i, l1 in enumerate(labels):
    for j, l2 in enumerate(labels):
        if i < j:
            r, _ = spearmanr(rdms[l1], rdms[l2])
            if r < 0.5:
                print(f"⚠  Low RDM consistency: {l1} ↔ {l2}: ρ = {r:.4f}")
                all_ok = False
if all_ok:
    print("✓ All within-WEIRD model pairs show ρ ≥ 0.5")

Spearman ρ — pairwise RDM correlation (WEIRD models, 10-term probe set)

                                    BGE-EN-large                E5-large              FreeLaw-EN
            BGE-EN-large                  1.0000                  0.3847                  0.3682
                E5-large                  0.3847                  1.0000                  0.2904
              FreeLaw-EN                  0.3682                  0.2904                  1.0000

⚠  Low RDM consistency: BGE-EN-large ↔ E5-large: ρ = 0.3847
⚠  Low RDM consistency: BGE-EN-large ↔ FreeLaw-EN: ρ = 0.3682
⚠  Low RDM consistency: E5-large ↔ FreeLaw-EN: ρ = 0.2904


## 6. Disk cache

The embedding cache stores computed arrays as `.npy` files keyed by a SHA-256
digest of the `(model_id, texts)` pair. This makes repeated experiment runs
fast without requiring GPU access after the first run.

The cell below forces a re-encode (`use_cache=False`), then reads the same
embeddings from disk (`use_cache=True`), and measures the speedup.

In [14]:
model_id = "BAAI/bge-large-en-v1.5"

t0 = time.perf_counter()
_ = client.embed(probe_en, model_id, use_cache=False)  # force re-encode
t_encode = time.perf_counter() - t0

t0 = time.perf_counter()
_ = client.embed(probe_en, model_id, use_cache=True)   # cache read
t_cache = time.perf_counter() - t0

print(f"Encode (no cache):  {t_encode:.3f}s")
print(f"Cache read:         {t_cache:.4f}s")
if t_cache > 0:
    print(f"Speedup:            {t_encode / t_cache:.0f}×")
print()
if client._cache_dir:
    nfiles = len(list(client._cache_dir.glob("*.npy")))
    print(f"Cache directory:    {client._cache_dir}")
    print(f"Cached .npy files:  {nfiles}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encode (no cache):  5.831s
Cache read:         0.0025s
Speedup:            2330×

Cache directory:    /Users/gpuzio/Desktop/CODE/THESIS/experiments/experiments/data/processed/embeddings_cache
Cached .npy files:  11


## Summary

| # | Check | Pass condition |
|---|-------|----------------|
| 1 | Shape + normalization | `vecs.shape == (N, dim)` and `‖v_i‖ = 1 ∀i` |
| 2 | Semantic coherence EN | `mean(legal↔legal) > mean(legal↔Swadesh)` for all 3 WEIRD models |
| 3 | Semantic coherence ZH | same for all 3 Sinic models |
| 4 | Within-WEIRD RDM consistency | Spearman ρ ≥ 0.5 for all 3 pairs |
| 5 | Disk cache | cache read faster than re-encode |

If all assertions pass and all ρ values are ≥ 0.5, `shared/embeddings.py`
is validated and ready for use in Lens I–V experiments.